# Amyloid pLM Pipeline — Kaggle (2× T4)

Runs the full pipeline on Kaggle using **both T4 GPUs**:
- **Stage 1 (embeddings):** ProtT5 on **GPU 0** and ESM2 on **GPU 1**, in parallel threads — the heavy PLM work uses both T4s at once.
- **Stages 2–4 (TensorFlow):** grid search → ensemble (5-fold CV) → benchmark, with an optional `MirroredStrategy` across both GPUs.

### Before running
1. **Settings → Accelerator → `GPU T4 x2`**.
2. **Settings → Internet → On** (to clone the repo + download the PLM weights).
3. **Add Input → your Dataset** containing the `New_dataset/` JSON files (training + benchmark).

Outputs are written to `/kaggle/working/results/<run_id>/{training,benchmark}/`.

## 0 · GPU check

In [ ]:
!nvidia-smi -L
import torch
n = torch.cuda.device_count()
print("torch", torch.__version__, "| visible CUDA GPUs:", n)
assert n >= 1, "No GPU. Settings -> Accelerator -> GPU T4 x2."
if n < 2:
    print("WARNING: only 1 GPU visible; embeddings will run single-GPU. Select 'GPU T4 x2'.")

## 1 · Clone repo & install deps

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/NGUYENTHAISON-2311/dl_plm_pipeline.git"
REPO_DIR = "/kaggle/working/dl_plm_pipeline"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
# Kaggle ships torch + tensorflow; ensure the embedding + IO extras are present.
subprocess.run("pip -q install 'transformers>=4.30' sentencepiece h5py pyyaml".split(),
               check=False)
print("repo ready:", REPO_DIR)

## 2 · Configure paths (Kaggle dataset in, /kaggle/working out)

In [ ]:
import glob
from src.utils.config import load_config, resolve_path
from src.utils.seed import set_global_seed

cfg = load_config()                       # repo config/default.yaml
set_global_seed(cfg.get("seed", 42))

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(
            f"{name} not found under /kaggle/input — attach the New_dataset files via "
            f"'Add Input' (Kaggle Dataset).")
    return hits[0]

cfg["paths"]["train_pos"]                = find("train_core_set_KA.json")
cfg["paths"]["train_neg"]                = find("disordered_regions_no_core_overlaps_filtered_clustered.json")
cfg["paths"]["classification_benchmark"] = find("new_representative_classification_benchmark_KA.json")
cfg["paths"]["position_benchmark"]       = find("new_representative_positive_benchmark_KA.json")

# Persisted Kaggle outputs.
cfg["embeddings"]["cache"]  = "/kaggle/working/embeddings.h5"
cfg["paths"]["results_dir"] = "/kaggle/working/results"
for k, v in cfg["paths"].items():
    print(f"{k:28s}: {v}")

## 3 · Run configuration

In [ ]:
import datetime
from pathlib import Path

RUN_GRIDSEARCH = True                      # False -> use each model's first grid combo (fast)
LIMIT          = None                      # e.g. 60 to smoke-test
MODELS         = cfg["gridsearch"]["models"]   # e.g. ["fnn", "cnn"] to go faster
EPOCHS         = cfg["train"]["epochs"]        # e.g. 12 for a quick run
USE_MIRRORED   = False                     # multi-GPU Keras training; models are tiny so rarely helps

cfg["gridsearch"]["models"] = MODELS
cfg["train"]["epochs"]      = EPOCHS
cfg["train"]["distribute"]  = "mirrored" if USE_MIRRORED else "none"

RUN_ID    = datetime.datetime.now().strftime("%y%m%d_%H%M%S")
run_dir   = Path(cfg["paths"]["results_dir"]) / RUN_ID
train_dir = run_dir / "training"
bench_dir = run_dir / "benchmark"
for d in (train_dir / "plots", bench_dir / "plots"):
    d.mkdir(parents=True, exist_ok=True)
cfg["paths"]["results_dir"] = str(train_dir)   # grid-search/ensemble artifacts land here

print("RUN_ID :", RUN_ID)
print("models :", MODELS, "| epochs:", EPOCHS, "| gridsearch:", RUN_GRIDSEARCH, "| mirrored:", USE_MIRRORED)
print("out    :", run_dir)

## 4 · Stage 1 — embeddings on **both T4 GPUs** (PyTorch)

ProtT5 → `cuda:0`, ESM2 → `cuda:1`, run in parallel threads. First run downloads the PLM
weights (~5 GB) and embeds every sequence; results are cached to `/kaggle/working/embeddings.h5`.

In [ ]:
from src.data.dataset import load_training_peptides, load_benchmark, assert_no_leakage
from src.data.embeddings import EmbeddingCache, generate_embeddings, generate_embeddings_dual_gpu
from src.utils.windows import required_inference_keys

peptides  = load_training_peptides(resolve_path(cfg, "train_pos"), resolve_path(cfg, "train_neg"))
benchmark = load_benchmark(resolve_path(cfg, "classification_benchmark"))
peptides  = assert_no_leakage(peptides, benchmark, drop=True)
if LIMIT:
    peptides = peptides[:LIMIT]
print(f"{len(peptides)} training peptides | {len(benchmark)} benchmark sequences")

cache    = EmbeddingCache(cfg["embeddings"]["cache"])
all_seqs = [p.sequence for p in peptides] + required_inference_keys([b.sequence for b in benchmark], cfg)
try:
    pos_bench = load_benchmark(resolve_path(cfg, "position_benchmark"))
    all_seqs += required_inference_keys([b.sequence for b in pos_bench], cfg)
except Exception:
    pass

import torch
if torch.cuda.device_count() >= 2:
    generate_embeddings_dual_gpu(all_seqs, cfg, "cuda:0", "cuda:1")   # <-- both T4s in parallel
else:
    cfg["embeddings"]["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    generate_embeddings(all_seqs, cfg)
print("embeddings cached at:", cache.path)

Free PyTorch GPU memory before TensorFlow takes over the GPUs.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("released torch CUDA memory")

## 5 · TensorFlow setup (memory growth so TF + the freed torch context coexist)

In [ ]:
import tensorflow as tf
for g in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
print("TF", tf.__version__, "| GPUs:", [d.name for d in tf.config.list_physical_devices("GPU")])

## 6 · Stage 2 — grid search (5-fold CV per architecture)

In [ ]:
import numpy as np
import pandas as pd
from src.training.grid_search import run_gridsearch, default_best_per_model
from src.utils.io import write_json

if RUN_GRIDSEARCH:
    best_per_model = run_gridsearch(cfg, cache, peptides)     # writes gridsearch.csv + best/
else:
    best_per_model = default_best_per_model(cfg)
    write_json(best_per_model, train_dir / "best_per_model.json")

pd.DataFrame([{"model": m, **v["best_hp"]} for m, v in best_per_model.items()])

## 7 · Stage 3 — ensemble (best arch × 5 folds) + out-of-fold predictions

In [ ]:
from src.training.ensemble import build_ensemble_cv

ensemble, oof, per_model_fold = build_ensemble_cv(cfg, cache, peptides, best_per_model)
ensemble.save(train_dir / "ensemble")
print(f"{len(ensemble.members)} ensemble members "
      f"({len(best_per_model)} architectures x {cfg['cv']['n_splits']} folds)")

## 8 · Training metrics — residue-level OOF (AUROC / PR / confusion)

In [ ]:
from src.evaluation.metrics import residue_metrics
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

thr = cfg["inference"]["residue_threshold"]
cv_df = pd.DataFrame(per_model_fold)
cv_summary = cv_df.groupby("model")[["auroc","auprc","mcc","f1","precision","recall"]].mean().round(3)
ens_metrics = residue_metrics(oof["labels"], oof["scores"], oof["mask"], threshold=thr)
print("Ensemble OOF residue metrics:")
for k, v in ens_metrics.items():
    print(f"  {k:10s}: {v}")

write_json({"ensemble_oof": ens_metrics,
            "per_model_fold": per_model_fold,
            "per_model_mean": cv_summary.reset_index().to_dict("records")},
           train_dir / "metrics.json")
cv_df.to_csv(train_dir / "per_model_fold_metrics.csv", index=False)
np.savez_compressed(train_dir / "oof_residue_scores.npz",
                    ids=np.array(oof["ids"]), scores=oof["scores"],
                    labels=oof["labels"], mask=oof["mask"])

m = oof["mask"].astype(bool)
y_true  = oof["labels"][m].ravel().astype(int)
y_score = oof["scores"][m].ravel()
y_pred  = (y_score > thr).astype(int)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_true, y_score, ax=ax[0]);        ax[0].set_title("ROC (residue OOF)")
PrecisionRecallDisplay.from_predictions(y_true, y_score, ax=ax[1]); ax[1].set_title("PR (residue OOF)")
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax[2], colorbar=False,
                                        display_labels=["non-core","core"]); ax[2].set_title("Confusion (residue OOF)")
plt.tight_layout(); fig.savefig(train_dir / "plots" / "training_metrics.png", dpi=120); plt.show()
cv_summary

## 9 · Stage 4 — benchmark (sliding window + >0.5 / run>10 rule)

In [ ]:
from src.evaluation.sliding import score_sequence_detailed
from src.evaluation.classify import classify_profile, sequence_score
from src.evaluation.metrics import sequence_metrics
from sklearn.metrics import roc_auc_score, average_precision_score

min_run = cfg["inference"]["min_consecutive"]
profiles, windows_out = {}, {}
y_true_seq, y_pred_seq, y_score_seq = [], [], []
for b in benchmark:
    profile, windows = score_sequence_detailed(b.sequence, ensemble, cache, cfg)
    label, run, _ = classify_profile(profile, thr, min_run)
    sscore = sequence_score(profile, min_run)
    y_true_seq.append(b.label); y_pred_seq.append(label); y_score_seq.append(sscore)
    profiles[b.id]    = {"label_true": b.label, "label_pred": label, "longest_run": int(run),
                         "sequence_score": round(float(sscore), 4),
                         "profile": [round(float(x), 4) for x in profile]}
    windows_out[b.id] = windows

seq_m = sequence_metrics(y_true_seq, y_pred_seq)
seq_m["auroc"] = float(roc_auc_score(y_true_seq, y_score_seq)) if len(set(y_true_seq)) > 1 else float("nan")
seq_m["auprc"] = float(average_precision_score(y_true_seq, y_score_seq))
print("Benchmark sequence-level metrics:")
for k, v in seq_m.items():
    print(f"  {k:10s}: {v}")

write_json({"sequence_level": seq_m,
            "inference": {"residue_threshold": thr, "min_consecutive": min_run,
                          "window_len": cfg["window_len"], "stride": cfg["inference"]["window_stride"]},
            "ensemble_members": len(ensemble.members)}, bench_dir / "metrics.json")
write_json(profiles, bench_dir / "profiles.json")
write_json(windows_out, bench_dir / "windows.json")
pd.DataFrame({"id": [b.id for b in benchmark], "true": y_true_seq,
              "pred": y_pred_seq, "score": y_score_seq}).to_csv(bench_dir / "sequence_predictions.csv", index=False)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_true_seq, y_score_seq, ax=ax[0]);        ax[0].set_title("ROC (sequence)")
PrecisionRecallDisplay.from_predictions(y_true_seq, y_score_seq, ax=ax[1]); ax[1].set_title("PR (sequence)")
ConfusionMatrixDisplay.from_predictions(y_true_seq, y_pred_seq, ax=ax[2], colorbar=False,
                                        display_labels=["NONAMYLOID","AMYLOID"]); ax[2].set_title("Confusion (sequence)")
plt.tight_layout(); fig.savefig(bench_dir / "plots" / "benchmark_metrics.png", dpi=120); plt.show()

## 10 · Summary — saved artifacts

In [ ]:
print("RUN_ID:", RUN_ID, "\n")
for tag, d in (("TRAINING", train_dir), ("BENCHMARK", bench_dir)):
    print(f"{tag}: {d}")
    for p in sorted(Path(d).rglob("*")):
        if p.is_file():
            print("   ", p.relative_to(d))
    print()
print("Download from the Kaggle 'Output' tab, or commit/save the notebook version to persist them.")